<a href="https://colab.research.google.com/github/aiinuuazzariaa/aiinuuazzariaa/blob/main/project_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To install Python libraries in Google Colab, you can use the `pip install` command. It's usually prefixed with an exclamation mark `!` to run it as a shell command within the notebook.

1. Install library

In [ ]:
!pip install openpyxl rapidfuzz pandas scikit-learn

2. Import library

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print("Semua library berhasil diimport!")

Semua library berhasil diimport!


3. Upload file spreadsheet

In [ ]:
uploaded = files.upload()

# Edit file name
filename = list(uploaded.keys())[0]
print(f"File diupload: {filename}")

# Read file
if filename.endswith('.csv'):
    df = pd.read_csv(filename)
else:
    df = pd.read_excel(filename)

print(f"Total baris: {len(df)}")
print(f"Kolom tersedia: {list(df.columns)}")
df.head()

4. Tentukan kolom yang dibandingkan

In [ ]:
# Edit column
KOLOM_A = 'DATA MASTER'
KOLOM_B = 'DATA SISTEM'

# Bobot algoritma (total harus = 1.0)
BOBOT_TFIDF = 0.6        # 60% TF-IDF
BOBOT_LEV   = 0.4        # 40% Levenshtein

# Threshold
THRESHOLD = 0.80

print(f"DATA MASTER : {KOLOM_A}")
print(f"DATA SISTEM : {KOLOM_B}")
print(f"Bobot : TF-IDF {BOBOT_TFIDF} | Levenshtein {BOBOT_LEV}")
print(f"Threshold: {THRESHOLD}")

DATA MASTER : DATA MASTER
DATA SISTEM : DATA SISTEM
Bobot : TF-IDF 0.6 | Levenshtein 0.4
Threshold: 0.8


6. Preprocessing function

In [ ]:
# Preprocessing
def preprocess(text):
  if pd.isna(text): return ""
  text = str(text).lower().strip()
  text = re.sub(r'[^\w\s]', ' ', text)
  text = re.sub(r'\s+', ' ', text).strip()
  return text

# Skor TF-IDF
def skor_tfidf(a, b):
  if not a or not b: return 0.0
  try:
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,3))
    matrix = vec.fit_transform([a, b])
    return float(cosine_similarity(matrix[0], matrix[1])[0][0])
  except:
    return 0.0

# Skor Levenshtein
def skor_levenshtein(a, b):
  if not a or not b: return 0.0
  return fuzz.ratio(a, b) / 100.0

# Hybrid function
def cek_kesamaan(raw_a, raw_b):
  a = preprocess(raw_a)
  b = preprocess(raw_b)
  s1 = skor_tfidf(a, b)
  s2 = skor_levenshtein(a, b)
  final = (BOBOT_TFIDF * s1) + (BOBOT_LEV * s2)
  status = "SAMA" if final >= THRESHOLD else "BEDA"
  return round(s1,3), round(s2,3), round(final,3), status

print("Semua fungsi siap digunakan!")

Semua fungsi siap digunakan!


7. Run data spreadsheet

In [ ]:
from openpyxl import load_workbook

KOLOM_STATUS = 'ACTION'

wb = load_workbook(filename)
ws = wb.active

header = {cell.value: cell.column for cell in ws[3] if cell.value}
col_a  = header[KOLOM_A]
col_b  = header[KOLOM_B]
col_st = header[KOLOM_STATUS]

sama = beda = dilewat = 0

for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
  val_a = row[col_a - 1].value
  val_b = row[col_b - 1].value

  if not val_a or not val_b:
    row[col_st - 1].value = None
    dilewat += 1
    continue

  if not str(val_a).strip() or not str(val_b).strip():
    row[col_st - 1].value = None
    dilewat += 1
    continue

  _, _, _, status = cek_kesamaan(val_a, val_b)
  row[col_st - 1].value = status
  if status == "SAMA": sama += 1
  else: beda += 1

wb.save(filename)

print(f"File '{filename}' berhasil diupdate!")
print(f"SAMA : {sama} baris")
print(f"BEDA : {beda} baris")
print(f"Dilewati : {dilewat} baris (kosong)")

KeyError: 'DATA MASTER'

8. Export result to excel

In [ ]:
files.download(filename)
print(f"✅ File '{filename}' berhasil didownload!")